# Lezione 17 — Sentence embeddings: un vettore per frase

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 16
(`Embedding` + `GlobalAveragePooling1D` dentro un classificatore). Lì il
pooling era un ingranaggio interno, invisibile: serviva solo a far
digerire la sequenza al `Dense` successivo. Oggi lo **estraiamo** come
prodotto a se' stante — un vettore che rappresenta l'intera frase, riusabile
per compiti diversi dalla classificazione.

**Cosa saprai fare alla fine:** distinguere un embedding di parola da un
embedding di frase, confrontare mean pooling e max pooling, e costruire una
funzione che trasforma qualunque testo di memoria in un vettore fisso,
pronta per la ricerca per similarita' delle prossime lezioni.

## Parte 1 — Teoria: dal vettore-parola al vettore-frase

Un **sentence embedding** e' un vettore di lunghezza fissa che rappresenta
il significato di un intero testo (una frase, o qui una memoria), non di
una singola parola. La Lezione 16 ne ha gia' costruito uno senza dirlo
esplicitamente: `GlobalAveragePooling1D` applicato ai vettori-parola di
`Embedding` produce esattamente questo — un vettore fisso per frase. La
differenza oggi e' d'intenzione: prima il vettore era un passo intermedio
verso una predizione; ora lo trattiamo come l'**output finale** che ci
interessa, da riusare per confrontare, cercare, raggruppare memorie
(prossime lezioni).

**Strategie di pooling.** Il mean pooling (Lezione 16) fa la media dei
vettori-parola: robusto, ma "diluisce" — una parola molto informativa in
una frase lunga pesa quanto le altre. Il **max pooling**
(`GlobalMaxPooling1D`) prende, per ciascuna delle `output_dim` dimensioni,
il valore massimo tra tutti i token della frase: cattura il segnale piu'
forte per ogni dimensione, ma e' piu' sensibile al rumore (un singolo
valore anomalo in un token domina quella dimensione). Nessuna delle due e'
"quella giusta" in assoluto: sono compromessi diversi, e in pratica si
sceglie confrontando i risultati (come farai nell'esercizio).

Un terzo approccio, molto piu' potente, e' pesare ogni vettore-parola in
base a *quanto e' rilevante per quella frase specifica* invece di
trattarli tutti allo stesso modo (media) o prenderne solo il picco
(massimo): e' l'idea dell'**attention**, il meccanismo alla base dei
Transformer — la vedrai nella Fase 5 del corso. Mean e max pooling sono i
punti di partenza storici; l'attention e' il passo successivo.

!!! info "Encoder di frase pre-addestrati, e perche' qui non li usiamo"
    In produzione, il modo piu' comune di ottenere sentence embedding di
    alta qualita' non e' addestrare un `Embedding` da zero come nella
    Lezione 16: si usano encoder **pre-addestrati** su miliardi di coppie
    di frasi (Universal Sentence Encoder, Sentence-BERT/
    `sentence-transformers`), che catturano somiglianza semantica molto
    meglio di quanto un dataset di poche centinaia di righe possa
    insegnare da zero. Questo ambiente non ha credenziali Kaggle/
    HuggingFace ne' GPU per scaricare ed eseguire quei modelli: qui
    costruiamo un sentence embedding "from scratch", piu' debole ma
    completamente trasparente — vedi *ogni* passo che lo produce, cosa
    che un encoder pre-addestrato nasconderebbe. Il principio (vettore
    denso di lunghezza fissa per frase, confrontabile con la similarita'
    coseno) e' identico.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import keras
from keras.layers import TextVectorization

keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

Keras 3.15.0


In [2]:
frasi_demo = np.array(['Marco visited Glasgow with his son.',
                       'The user prefers short updates over long emails.'])

vettorizzatore_demo = TextVectorization(max_tokens=20, output_mode='int',
                                        output_sequence_length=10)
vettorizzatore_demo.adapt(frasi_demo)
sequenze_demo = vettorizzatore_demo(frasi_demo)

embedding_demo = keras.layers.Embedding(input_dim=20, output_dim=4)
vettori_parola_demo = embedding_demo(sequenze_demo)

mean_pool = keras.layers.GlobalAveragePooling1D()(vettori_parola_demo)
print('mean pooling, prima frase:', mean_pool[0].numpy())

mean pooling, prima frase: [ 0.01903404 -0.00204391 -0.01410179  0.02161337]


2026-07-16 21:16:14.273482: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Parte 2 — Esercizio guidato

Il tuo compito: calcola il **max pooling** di `vettori_parola_demo` con
`keras.layers.GlobalMaxPooling1D()` e confronta il vettore risultante con
`mean_pool[0]` stampato sopra. Sono uguali? Perche' no?

In [3]:
# Scrivi qui: applica keras.layers.GlobalMaxPooling1D() a vettori_parola_demo
# e stampa il vettore della prima frase.

pass

### Soluzione spiegata

Mean e max pooling leggono la stessa sequenza di vettori-parola in due
modi diversi: la media "spalma" il contributo di ogni token, il massimo
isola il valore piu' alto per ciascuna dimensione. Sono quasi sempre
diversi (a meno di sequenze degeneri, es. una sola parola non-padding),
perche' rispondono a domande diverse — "qual e' il valore tipico?" contro
"qual e' il valore piu' estremo?" — sugli stessi dati.

In [4]:
max_pool = keras.layers.GlobalMaxPooling1D()(vettori_parola_demo)
print('mean pooling, prima frase:', mean_pool[0].numpy())
print('max pooling,  prima frase:', max_pool[0].numpy())
assert not np.allclose(mean_pool[0].numpy(), max_pool[0].numpy())

mean pooling, prima frase: [ 0.01903404 -0.00204391 -0.01410179  0.02161337]
max pooling,  prima frase: [0.04660337 0.01626286 0.02913921 0.04891639]


## Parte 3 — Il progetto: Memory AI Lab, passo 17 — un incorporatore di memorie

Costruiamo un piccolo classificatore come nella Lezione 16 (serve per
addestrare l'`Embedding`: senza un compito da imparare, i pesi restano
casuali), ma questa volta lo scopo non e' la sua accuratezza — e' il
vettore che produce **prima** dell'ultimo `Dense`. Estraiamo quel vettore
con un `keras.Model` che condivide i pesi del classificatore ma si ferma
al pooling: un **incorporatore di memorie** (`incorpora(testi)`) che
useremo nelle prossime lezioni per cercare memorie simili, raggrupparle e
misurare la qualita' della ricerca. Come nella Lezione 16,
`Embedding(..., mask_zero=True)`: senza, il pooling medierebbe anche il
padding, che con frasi brevi in sequenze da 24 token e' spesso la
maggioranza — il vettore-frase finale rischia di somigliare piu' al
padding che al contenuto.

In [5]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

LUNGHEZZA_SEQUENZA = 24
vettorizzatore = TextVectorization(max_tokens=300, output_mode='int',
                                   output_sequence_length=LUNGHEZZA_SEQUENZA)
vettorizzatore.adapt(testi['train'])
X_seq = {n: vettorizzatore(t).numpy() for n, t in testi.items()}

In [6]:
keras.utils.set_random_seed(42)
ingresso = keras.Input(shape=(LUNGHEZZA_SEQUENZA,))
parole = keras.layers.Embedding(input_dim=300, output_dim=16, mask_zero=True)(ingresso)
vettore_frase = keras.layers.GlobalAveragePooling1D(name='sentence_embedding')(parole)
nascosto = keras.layers.Dense(32, activation='relu')(vettore_frase)
nascosto = keras.layers.Dropout(0.3)(nascosto)
uscita = keras.layers.Dense(4, activation='softmax')(nascosto)

classificatore = keras.Model(ingresso, uscita)
classificatore.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                       metrics=['accuracy'])
classificatore.fit(X_seq['train'], target['train'], epochs=300,
                   validation_data=(X_seq['val'], target['val']),
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                            restore_best_weights=True)],
                   verbose=0)
acc = classificatore.evaluate(X_seq['val'], target['val'], verbose=0)[1]
print(f'accuratezza validation: {acc:.0%} (come riferimento, non e\' il punto della lezione)')

accuratezza validation: 95% (come riferimento, non e' il punto della lezione)


In [7]:
# L'incorporatore: stesso modello, ma l'output e' il vettore-frase, non la predizione.
incorporatore = keras.Model(classificatore.input,
                            classificatore.get_layer('sentence_embedding').output)

def incorpora(testi_np):
    sequenze = vettorizzatore(testi_np)
    return incorporatore(sequenze).numpy()

esempio = memorie['val']['text'].astype(str).to_numpy()[:3]
vettori = incorpora(esempio)
print('forma:', vettori.shape, '(memorie x dimensione embedding)')
for testo, v in zip(esempio, vettori):
    print(f'- "{testo[:60]}..." -> norma {np.linalg.norm(v):.3f}')

# Determinismo: lo stesso testo produce sempre lo stesso vettore.
ripetuto = incorpora(esempio[:1])
assert np.allclose(vettori[0], ripetuto[0])

forma: (3, 16) (memorie x dimensione embedding)
- "Lucia works on la riunione settimanale...." -> norma 0.403
- "Elena met a friend in Roma...." -> norma 0.540
- "Marco visited Glasgow for the weekend...." -> norma 0.618


## Cosa hai imparato

- Un **sentence embedding** e' un vettore fisso che rappresenta un intero
  testo; mean pooling (Lezione 16) e' gia' un modo di produrne uno, qui lo
  trattiamo come prodotto finale invece che passo intermedio.
- **Mean pooling** e **max pooling** sono due compromessi diversi per
  aggregare vettori-parola in un vettore-frase; l'**attention**
  (Transformer, Fase 5) e' un'evoluzione che pesa i token invece di
  mediarli o prenderne il massimo.
- Un `keras.Model` puo' condividere i pesi di un classificatore ma
  fermarsi a un layer intermedio: cosi' si trasforma un modello di
  classificazione in un **incorporatore** riusabile.
- Gli encoder di frase pre-addestrati (USE, Sentence-BERT) esistono e sono
  piu' potenti; qui restiamo "from scratch" per trasparenza e per i
  limiti di questo ambiente (nessuna credenziale, nessuna GPU).

## Quiz

1. In cosa differisce concettualmente l'uso del pooling qui rispetto alla
   Lezione 16, anche se il calcolo (`GlobalAveragePooling1D`) e'
   identico?
2. Perche' max pooling e mean pooling danno quasi sempre vettori diversi
   sulla stessa frase?
3. Perche' l'incorporatore e il classificatore condividono gli stessi pesi
   di `Embedding`, invece di essere due modelli allenati separatamente?

<details>
<summary><b>Apri le risposte</b></summary>

1. Nella Lezione 16 il pooling era un passo interno di una pipeline che
   finiva con una predizione; qui lo stesso calcolo produce l'**output
   finale** che ci interessa — il vettore-frase in se', da riusare per
   compiti diversi dalla classificazione (similarita', clustering,
   retrieval).
2. Perche' rispondono a domande diverse sugli stessi vettori-parola: la
   media riflette il valore "tipico" lungo la frase, il massimo isola il
   valore piu' estremo per ciascuna dimensione — solo in casi degeneri
   (es. una frase con un solo token utile) coincidono.
3. Perche' l'`Embedding` va addestrato su un compito per non restare
   inizializzato a caso (Lezione 16): il classificatore fornisce quel
   compito. L'incorporatore riusa **esattamente** gli stessi pesi appresi,
   fermandosi prima dell'ultimo `Dense` — non e' un modello diverso, e' lo
   stesso modello osservato a un punto diverso.
</details>

## Fonti

- Keras, `GlobalMaxPooling1D`:
  https://keras.io/api/layers/pooling_layers/global_max_pooling1d/
- Keras, *The Functional API* (costruire modelli con output intermedi
  condivisi):
  https://keras.io/guides/functional_api/
- TensorFlow Hub, *Universal Sentence Encoder* (encoder pre-addestrato,
  citato per contesto, non usato in questo notebook):
  https://www.tensorflow.org/hub/tutorials/semantic_similarity_with_tf_hub_universal_encoder

## Prossima lezione

Ora che ogni memoria e' un vettore, la domanda naturale e': quanto sono
**simili** due vettori? La prossima lezione introduce la similarita'
coseno, il modo standard di rispondere.